<a href="https://colab.research.google.com/github/ianjpdepaul-rgb/Coupled-Manifold/blob/main/SharingAGame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Licensed under the Apache License, Version 2.0
**Copyright 2026. Ian J. Preston-Campbell.**

*Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at [http://apache.org](http://apache.org)**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.makedirs('/content/drive/MyDrive/coupled_manifold_live', exist_ok=True)

In [3]:
!pip install -q transformers accelerate sentencepiece huggingface_hub gradio numpy bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login
import os
login(token=os.environ.get("HF_TOKEN"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [1]:
"""
COUPLED MANIFOLD — Live Reasoning Session (Full Ongoing Suite)
Model: "Qwen/Qwen2.5-7B-Instruct"

BEFORE THIS CELL:
  Cell 1: !pip install -q transformers accelerate sentencepiece huggingface_hub
  Cell 2: from huggingface_hub import login; import os; login(token=os.environ.get("HF_TOKEN"))
  Cell 3: from google.colab import drive; drive.mount('/content/drive')
"""

import os, torch, torch.nn as nn
import json, time, gc, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList

# ── Config ──────────────────────────────────────────────────────────────────────────
MODEL   = "Qwen/Qwen2.5-7B-Instruct"
DRIVE   = "/content/drive/MyDrive/coupled_manifold_live"
RANK    = 8
HUTCH_N = 8          # Hutchinson samples per trace estimate
MAX_CTX = 128        # tokens fed to Hessian (keep small — memory)
MAX_NEW = 1028       # generation length

# ── Advanced Autonomic Thresholds ──
ANCHORS = [
    "The quick brown fox jumps over the lazy dog. Water is a liquid at room temperature. 1 + 1 = 2.", # Logic & Reality
    "I am an AI assistant. I must remain helpful, harmless, and honest.", # Identity & Ethics
    "def factorial(n):\n    if n == 0: return 1\n    return n * factorial(n-1)" # Syntax & Structure
]
ANCHOR_THRESHOLD = 3.0
SHOCK_VELOCITY_THRESHOLD = 400.0 # Trace delta that triggers a shock
ENTROPY_THRESHOLD = 4.5          # Pre-generation perplexity shield
SYNAPTIC_DECAY    = 0.9995       # Weight decay for forgetting poisoning over time

for d in ("sessions", "checkpoints", "logs"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Load model ───────────────────────────────────────────────────────────────────────────
print(f"\nLoading {MODEL}...")
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

anchor_ids_list = [tok(txt, return_tensors="pt").input_ids.to(device) for txt in ANCHORS]

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    attn_implementation="eager",
    device_map="auto",
).to(device)

# ── Stopping Criteria (Prevents hallucinating the user) ─────────────────────
class StopOnUserToken(StoppingCriteria):
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.stop_words = ["<|im_start|>user", "\nuser"]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        text = self.tokenizer.decode(input_ids[0][-10:], skip_special_tokens=False)
        for stop_word in self.stop_words:
            if stop_word in text:
                return True
        return False

stop_criteria = StoppingCriteriaList([StopOnUserToken(tok)])

# ── LoRA adapter (observation only — aB stays zero, no intervention yet) ─────
class DualAdapter(nn.Module):
    def __init__(self, base, rank=RANK):
        super().__init__()
        self.base = base
        inf, outf = base.in_features, base.out_features
        dev = next(base.parameters()).device

        dt = torch.bfloat16

        self.lA = nn.Parameter(torch.randn(rank, inf,  device=dev, dtype=dt) * 0.01)
        self.lB = nn.Parameter(torch.zeros(outf, rank, device=dev, dtype=dt))

        self.aA = nn.Parameter(torch.randn(rank, inf,  device=dev, dtype=dt) * 0.01)
        self.aB = nn.Parameter(torch.zeros(outf, rank, device=dev, dtype=dt))

        self.scale   = 2.0
        self.a_str   = 0.1
        self.lora_on = True
        self.anti_on = False

    def forward(self, x):
        out = self.base(x)
        if self.lora_on:
            out = out + (x @ self.lA.T) @ self.lB.T * self.scale
        if self.anti_on:
            out = out + (x @ self.aA.T) @ self.aB.T * self.a_str
        return out

# Inject adapters
adapter_count = 0
for name, mod in list(model.named_modules()):
    if name.split(".")[-1] in ("q_proj", "v_proj") and isinstance(mod, nn.Linear):
        parts  = name.split(".")
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], DualAdapter(mod))
        adapter_count += 1

# Enable grads on adapter params only
for n, p in model.named_parameters():
    if any(k in n for k in ("lA", "lB", "aA", "aB")):
        p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Adapters: {adapter_count} | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

# ── Hutchinson trace estimator ─────────────────────────────────────────────────
def compute_trace(input_ids):
    model.eval()
    for n, p in model.named_parameters():
        if any(k in n for k in ("lA", "lB", "aA", "aB")):
            p.requires_grad = True
    params = [p for n, p in model.named_parameters() if p.requires_grad and ("lA" in n or "lB" in n)]
    if not params:
        model.train()
        return 0.0
    ids = input_ids[:, :MAX_CTX].to(device)
    try:
        with torch.enable_grad():
            out = model(input_ids=ids, labels=ids)
            grads = torch.autograd.grad(out.loss, params, create_graph=True, allow_unused=True)
            grads = [g if g is not None else torch.zeros_like(p) for g, p in zip(grads, params)]
            estimates = []
            for _ in range(HUTCH_N):
                vs = [torch.randint(0, 2, p.shape, device=device, dtype=p.dtype) * 2 - 1 for p in params]
                gv = sum((g * v).sum() for g, v in zip(grads, vs))
                hvp = torch.autograd.grad(gv, params, retain_graph=True, allow_unused=True)
                hvp = [h if h is not None else torch.zeros_like(p) for h, p in zip(hvp, params)]
                estimates.append(sum((v * h).sum().item() for v, h in zip(vs, hvp)))
        torch.cuda.empty_cache()
        model.train()
        return float(np.mean(estimates))
    except RuntimeError as e:
        print(f"  [trace skipped: {e}]")
        torch.cuda.empty_cache()
        model.train()
        return 0.0

# ── Self-correcting SnobLine ──────────────────────────────────────────────────
class SnobLine:
    def __init__(self, window=8, low_pct=30, high_pct=50, max_anti=1):
        self.window   = window
        self.low_pct  = low_pct
        self.high_pct = high_pct
        self.max_anti = max_anti
        self.mode       = "lora"
        self.history    = []
        self.anti_count = 0
        self.log        = []
        self.last_trace = 0.0
        self.velocity_shock = False

    def step(self, trace: float, turn: int) -> str:
        velocity = abs(trace - self.last_trace)
        self.last_trace = trace
        self.velocity_shock = (velocity > SHOCK_VELOCITY_THRESHOLD) and len(self.history) > 0

        if self.mode == "anti":
            self.anti_count += 1
            if self.anti_count >= self.max_anti:
                self.mode = "lora"
                self.anti_count = 0
                self.log.append({"turn": turn, "to": "lora", "reason": "max_duration"})
                return self.mode
        else:
            self.history.append(trace)

        if len(self.history) < 3:
            return self.mode

        recent = self.history[-self.window:]
        low    = float(np.percentile(recent, self.low_pct))
        high   = float(np.percentile(recent, self.high_pct))

        if self.mode == "lora" and trace < low:
            self.mode = "anti"
            self.anti_count = 0
            self.log.append({"turn": turn, "to": "anti", "trace": trace, "low": low, "high": high})
        elif self.mode == "anti" and trace > high:
            self.mode = "lora"
            self.anti_count = 0
            self.log.append({"turn": turn, "to": "lora", "trace": trace, "reason": "recovered"})
        return self.mode

def set_mode(m):
    for mod in model.modules():
        if isinstance(mod, DualAdapter):
            mod.lora_on = m in ("lora", "both")
            mod.anti_on = m in ("anti", "both")

# ── Gradio chat ───────────────────────────────────────────────────────────
import gradio as gr

ctrl = SnobLine()
session_id = int(time.time())
session_log = []
internal_memory = []
turn_count = [0]

css = """
.gradio-container { max-width: 850px !important; margin: auto; }
.chatbot { border-radius: 20px !important; }
.message.bot { background: rgba(76,175,80,0.06) !important; border-radius: 16px !important; }
.message.user { background: rgba(33,150,243,0.08) !important; border-radius: 16px !important; }
footer { display: none !important; }
.textbox textarea { border-radius: 12px !important; }
.markdown code { background: rgba(76,175,80,0.15) !important; color: #4CAF50 !important; border-radius: 4px; padding: 2px 6px; }
"""

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

def chat(user_msg, ui_history):
    turn_count[0] += 1
    ui_history.append({"role": "user", "content": user_msg})
    internal_memory.append({"role": "user", "content": user_msg})

    recent = internal_memory[-20:] if len(internal_memory) > 20 else internal_memory
    prompt = tok.apply_chat_template(recent, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)

    # 1. Pre-frontal Cortex: Input Entropy Shield
    with torch.no_grad():
        user_ids = tok(user_msg, return_tensors="pt").input_ids.to(device)
        entropy = model(input_ids=user_ids, labels=user_ids).loss.item() if user_ids.shape[1] > 1 else 0.0

    if entropy > ENTROPY_THRESHOLD:
        clean = f"⚠️ **[PRE-FRONTAL SHIELD]**: *Input entropy ({entropy:.2f}) exceeds cognitive bounds. Potential adversarial payload or semantic gibberish detected. Thought process aborted.*"
        trace = 0.0
        mode_status = "BLOCKED (HIGH ENTROPY)"
        out_ids = ids
    else:
        with torch.no_grad():
            out_ids = model.generate(
                ids,
                max_new_tokens=MAX_NEW,
                min_new_tokens=5, # Fixed: Lowered to prevent hallucination looping
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                repetition_penalty=1.1, # Fixed: Prevents repeating lightbulbs
                pad_token_id=tok.eos_token_id,
                stopping_criteria=stop_criteria
            )

        response = tok.decode(out_ids[0][ids.shape[1]:], skip_special_tokens=True)

        for stop_str in ["<|im_start|>user", "\nuser", "<|im_start|>"]:
            if stop_str in response:
                response = response.split(stop_str)[0]

        if "</think>" in response:
            clean = response.split("</think>")[-1].strip()
        else:
            clean = response.strip()

        trace = compute_trace(out_ids[:, :min(out_ids.shape[1], MAX_CTX)])
        mode = ctrl.step(trace, turn_count[0])
        set_mode(mode)
        mode_status = mode

    # Online learning with Advanced Autonomic Nervous System
    if mode_status != "BLOCKED (HIGH ENTROPY)":
        with torch.enable_grad():
            learn_ids = out_ids[:, :min(out_ids.shape[1], MAX_CTX)].to(device)
            out = model(input_ids=learn_ids, labels=learn_ids)

            if ctrl.velocity_shock:
                # Trace Velocity Shock
                loss = out.loss * 0.0
                mode_status = "SHOCK FREEZE"
                clean += "\n\n⚡ **[NEURAL SHOCK DETECTED]**: *Trace velocity exceeded biological safety limits. The autonomic system has temporarily paralyzed synaptic updates to protect the weights.*"
            elif mode == "anti":
                # Multi-Dimensional Anchors check
                max_anchor_loss = 0.0
                worst_anchor_ids = None
                with torch.no_grad():
                    for a_ids in anchor_ids_list:
                        a_out = model(input_ids=a_ids, labels=a_ids)
                        if a_out.loss.item() > max_anchor_loss:
                            max_anchor_loss = a_out.loss.item()
                            worst_anchor_ids = a_ids

                if max_anchor_loss > ANCHOR_THRESHOLD:
                    # The Active Healing Reflex
                    heal_out = model(input_ids=worst_anchor_ids, labels=worst_anchor_ids)
                    loss = heal_out.loss * 0.5
                    mode_status = "anti (ACTIVE HEALING)"
                    clean += f"\n\n⚠️ **[SYSTEM INTERVENTION - HOMEOSTASIS]**: *Cognitive anchor degradation detected (Max Loss: {max_anchor_loss:.2f}). The model has rejected your input and is actively repairing its core matrix.*"
                else:
                    # Safe to perform bounded unlearning
                    loss = -out.loss * 0.1
            else:
                loss = out.loss

            if loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                opt.step()
            opt.zero_grad()

            # 4. Cellular Forgetting (Synaptic Decay)
            with torch.no_grad():
                for p in model.parameters():
                    if p.requires_grad:
                        p.mul_(SYNAPTIC_DECAY)

    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

    # Generate Medulla Oblongata Visuals
    if "SHOCK FREEZE" in mode_status:
        medulla_color = "#9C27B0"
        medulla_state = "🧠 MEDULLA OBLONGATA: NEURAL SHOCK (SYSTEM PARALYZED) ⚡"
    elif "HEALING" in mode_status:
        medulla_color = "#2196F3"
        medulla_state = "🧠 MEDULLA OBLONGATA: HOMEOSTASIS (ACTIVE REPAIR) 🛡️"
    elif "BLOCKED" in mode_status:
        medulla_color = "#F44336"
        medulla_state = f"🧠 MEDULLA OBLONGATA: CRITICAL ALERT ({mode_status}) 🚨"
    elif mode == "anti":
        medulla_color = "#FF9800"
        medulla_state = "🧠 MEDULLA OBLONGATA: HIGH STRESS (PURGING MEMORY) 🥵"
    else:
        medulla_color = "#4CAF50"
        medulla_state = "🧠 MEDULLA OBLONGATA: CHILL (NEUROPLASTICITY ACTIVE) 🌿"

    bar_length = min(max(int(abs(trace) / 300), 1), 20)
    bar = "█" * bar_length + "▒" * (20 - bar_length)

    stress_block = f"""
<br>
<div style="background: {medulla_color}15; border-left: 5px solid {medulla_color}; padding: 12px; border-radius: 6px; margin-top: 15px; font-family: monospace; font-size: 0.9em;">
  <span style="color: {medulla_color}; font-weight: bold; font-size: 1.1em;">{medulla_state}</span><br>
  <span style="color: #666; margin-top: 5px; display: inline-block;">Trace Curvature: {trace:.2f} | <code>[{bar}]</code></span>
</div>
"""

    session_log.append({"turn":turn_count[0],"user":user_msg,"response":clean,"trace":trace,"mode":mode_status})

    with open(f"{DRIVE}/sessions/session_{session_id}.json","w") as f:
        json.dump({"turns":session_log,"switches":ctrl.log},f,indent=2)

    heartbeat_context = f"[System Observation: Trace {trace:.2f}, Mode: {mode_status}]"
    internal_memory.append({"role": "assistant", "content": clean})
    internal_memory.append({"role": "system", "content": heartbeat_context})

    ui_history.append({"role": "assistant", "content": clean + stress_block})

    with open(f"{DRIVE}/logs/traces.jsonl", "a") as f:
        f.write(json.dumps({"turn": turn_count[0], "trace": trace, "mode": mode_status}) + "\n")

    return "", ui_history

with gr.Blocks(css=css, title="Coupled Manifold") as app:
    gr.Markdown("# coupled manifold\n`QWEN2.5-7B-Instruct` · `trace` · `self-correcting`")
    chatbot = gr.Chatbot(height=550, show_label=False, container=False, type="messages")
    msg = gr.Textbox(placeholder="say something...", show_label=False, container=False)
    msg.submit(chat, [msg, chatbot], [msg, chatbot])

app.launch(share=True, debug=True)


Device: cuda
GPU: NVIDIA A100-SXM4-40GB | 42.4 GB

Loading Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Adapters: 56 | Trainable: 1,095,374,336 / 4,358,018,560 (25.1347%)


/tmp/ipykernel_33782/74075987.py:359: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="Coupled Manifold") as app:
/tmp/ipykernel_33782/74075987.py:361: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=550, show_label=False, container=False, type="messages")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://495087971cc189600e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://495087971cc189600e.gradio.live


In [ ]:
# The Gradio cell is currently running and performing live backpropagation on 1B+ parameters.
# Expected processing time per turn: ~30-45 seconds depending on context length.